In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load shapefiles
tx_tracts = gpd.read_file('./tract_shapefiles/tl_2023_tx_tract/tl_2023_48_tract.shp')

# === Load station data and extract lat/lon ===
austin_stations = pd.read_csv('./bikeshare_data/austin_bikeshare/austin_bikeshare_stations_v2.csv')

# Parse the "(lat, lon)" string into separate columns
austin_stations[["latitude", "longitude"]] = (
    austin_stations["Location"]
    .str.strip("()")
    .str.split(",", expand=True)
    .apply(lambda col: col.str.strip())
    .astype(float)
)

austin_trips = pd.read_csv('./bikeshare_data/austin_bikeshare/austin_bikeshare_trips_v2.csv')


In [ ]:
austin_stations.columns

In [ ]:
austin_trips.columns

In [ ]:
austin_stations.columns

In [ ]:
austin_trips['Checkout Datetime']

In [ ]:
austin_trips['Checkout Date'].min()

In [ ]:
# 1. Convert austin_stations to a GeoDataFrame
stations_gdf = gpd.GeoDataFrame(
    austin_stations,
    geometry=[Point(xy) for xy in zip(austin_stations.longitude, austin_stations.latitude)],
    crs="EPSG:4326"
)

# 2. Align CRS with tx_tracts
stations_gdf = stations_gdf.to_crs(tx_tracts.crs)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,8))
tx_tracts.plot(ax=ax, color='white', edgecolor='black')
stations_gdf.plot(ax=ax, marker='o', color='red', markersize=5)
plt.show()

# 3. Spatial join to tag stations with tract GEOID and ALAND
tracts_subset = tx_tracts[['GEOID', 'ALAND', 'geometry']]
stations_with_tract = gpd.sjoin(stations_gdf, tracts_subset, how='left', predicate='within')
station_to_tract = stations_with_tract[['Kiosk ID', 'GEOID']].set_index('Kiosk ID')

# 4. Map trips to checkout and return tracts
trip_df = austin_trips.copy()
trip_df['checkout_tract'] = trip_df['Checkout Kiosk ID'].map(station_to_tract['GEOID'])
trip_df['return_tract'] = trip_df['Return Kiosk ID'].map(station_to_tract['GEOID'])

# 5. Aggregate trip counts by tract
trips_out = trip_df.groupby('checkout_tract').size().rename('trips_out')
trips_in  = trip_df.groupby('return_tract').size().rename('trips_in')

# 6. Merge counts with all tracts and compute total
tract_counts = pd.DataFrame({
    'GEOID': tx_tracts['GEOID'],
    'ALAND': tx_tracts['ALAND']
}).set_index('GEOID')
tract_counts = tract_counts.join(trips_out).join(trips_in).fillna(0)
tract_counts['total_trips'] = tract_counts['trips_out'] + tract_counts['trips_in']
tract_counts = tract_counts[tract_counts['total_trips'] > 0]
tract_counts = tract_counts.reset_index()

# 3651 days between 01/01/2014 and 12/31/2023
tract_counts['trips_per_day'] = tract_counts['total_trips'] / 3651
tract_counts = tract_counts.drop(columns=['trips_in', 'trips_out'])
# Display the resulting DataFrame
tract_counts.to_csv('./bikeshare_data/tracts_with_trips/austin_tracts_with_trips.csv')

In [ ]:
austin_csv_files = ['./austin_agg_census/tx_transport_main.csv', 
                './austin_agg_census/selected_econ_dp03.csv', 
                './austin_agg_census/geo_mobility_s0701.csv',
                './austin_agg_census/tx_sex_by_commute_s0801.csv',
                './total_pop_s0601.csv'
                ,'./full_s0101.csv'
                ]

variable_meanings = {}

merged_austin_data = pd.read_csv(austin_csv_files[0])
merged_austin_data['GEOID'] = merged_austin_data['GEO_ID'].str.split('US').str[1]  # Extract GEOID from GEO_ID
merged_austin_data = merged_austin_data.drop(columns=['GEO_ID'])  # Drop redundant column

meanings = merged_austin_data.iloc[0].to_dict()  # Convert the first row to a dictionary
variable_meanings.update(meanings)  # Update the global dictionary of meanings
merged_austin_data = merged_austin_data.iloc[1:].reset_index(drop=True)

# Sequentially merge the remaining CSVs
for file in austin_csv_files[1:]:
    df = pd.read_csv(file)
    df['GEOID'] = df['GEO_ID'].str.split('US').str[1]  # Extract GEOID from GEO_ID
    df = df.drop(columns=['GEO_ID'])  # Drop redundant column
    df = df.drop(columns=['NAME'])  # Drop redundant column
    meanings = df.iloc[0].to_dict()  # Convert the first row to a dictionary
    variable_meanings.update(meanings)  # Update the global dictionary of meanings
    df = df.iloc[1:].reset_index(drop=True)

    merged_austin_data = pd.merge(merged_austin_data, df, on='GEOID', how='outer')  # Merge on GEOID


# Merge with tracts_with_trips DataFrame
merged_df = pd.merge(tracts_with_trips, merged_austin_data, on='GEOID', how='left')

# Ensure all object columns are converted to numeric where possible
for col in merged_df.select_dtypes(include=['object']).columns:
    merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')


# Exclude columns whose names end with M
columns_to_exclude = [col for col in merged_df.columns if col.endswith('M')]
filtered_columns = merged_df.columns.difference(columns_to_exclude)

# Select numeric columns that are not excluded
numeric_columns = merged_df[filtered_columns].select_dtypes(include=['number']).columns
merged_df = merged_df[numeric_columns]

merged_df['trips_per_day'] = merged_df['total_trips'] / 1278

# List of columns to transform
columns_to_transform = [col for col in merged_df.columns if col not in ['GEOID', 'ALAND', 'total_trips', 'trips_per_day']]

# Creating new columns by dividing by population and area
for col in columns_to_transform:
    merged_df[f'{col}_per_capita'] = merged_df[col] / merged_df['S0101_C01_001E'] # total population
    merged_df[f'{col}_per_ALAND'] = merged_df[col] / merged_df['ALAND']

# Save combined dataframe for Multiple Regression analysis
merged_df.to_csv('./multiple_regression/austin_census_data_and_trips.csv')

# Calculate correlation between bike trips and all other numeric columns
correlations = merged_df.corrwith(merged_df['total_trips'])
#correlations = merged_df[numeric_columns].corrwith(merged_df['total_trips'] / merged_df['ALAND'])

# Convert to a DataFrame for better readability and sort by absolute correlation
correlation_df = correlations.reset_index()
correlation_df.columns = ['Variable', 'Correlation']
correlation_df['Absolute_Correlation'] = correlation_df['Correlation'].abs()

# Sort by absolute correlation in descending order
sorted_correlation_df = correlation_df.sort_values(by='Absolute_Correlation', ascending=False)

# Extracting the relevant prefix for variable names
def extract_prefix(variable_name):
    for i, char in enumerate(variable_name):
        if char.upper() == 'E':
            return variable_name[:i + 1]
    return variable_name  # Return the whole variable name if 'E' is not found

# Apply the function to create a new column 'Meaning'
sorted_correlation_df['Meaning'] = sorted_correlation_df['Variable'].apply(lambda x: variable_meanings.get(extract_prefix(x), 'Unknown'))

# Display the top variables
columns_to_print = ['Meaning', 'Correlation']
print(sorted_correlation_df[columns_to_print])


In [ ]:
output_path = './correlation_results/austin_correlations.csv'
sorted_correlation_df.to_csv(output_path, index=False)